<a href="https://colab.research.google.com/github/alibarro/colab/blob/main/prospectivity_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mineral Prospectivity Mapping Workflow

Data-driven mineral prospectivity mapping using **Landsat 8** optical imagery, **Sentinel-1** SAR-derived structural lineaments, **SRTM** topography, and a **Random Forest** classifier — run on **Google Earth Engine** via `geemap`.

**Pipeline**
1. Setup & authentication
2. Configuration (AOI, date ranges, model hyperparameters)
3. Landsat 8 acquisition & cloud masking
4. Topographic derivatives (SRTM)
5. Sentinel-1 structural lineament extraction
6. Spectral mineral-alteration proxies
7. Feature stack assembly
8. Random Forest training + held-out accuracy assessment
9. Map visualization
10. Optional export to Google Drive

> **Note:** The training points below are illustrative placeholders. Replace `POSITIVE_SITES` with verified deposit/occurrence coordinates before drawing real conclusions. Background ("pseudo-absence") points are randomly sampled, not verified barren ground — treat outputs as a relative ranking, not a validated presence/absence prediction.


## 1. Setup & Authentication

Run this once per Colab session.

In [2]:
# Install/upgrade required packages (Colab ships an old geemap by default)
!pip install -q -U geemap earthengine-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.6/481.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.
google-genai 2.12.1 requires google-auth[requests]<2.56.0,>=2.48.1, but you have google-auth 2.57.1 which is incompatible.


In [3]:
import ee
import geemap

# Authenticate then initialize. On first run this opens a browser flow;
# replace 'your-gcp-project-id' with a Google Cloud project that has the
# Earth Engine API enabled.
ee.Authenticate()
ee.Initialize(project='ee-barroali')

## 2. Imports & Logging

In [4]:
from __future__ import annotations

import logging
from dataclasses import dataclass, field
from typing import Sequence

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("prospectivity")

## 3. Configuration

All tunable parameters live in one dataclass, including the area of interest (AOI).

In [5]:
@dataclass(frozen=True)
class ProspectivityConfig:
    """Central configuration for the prospectivity mapping run."""

    # Area of interest: must be defined by the caller (ee.Geometry / FeatureCollection)
    aoi: ee.Geometry

    # Optical (Landsat 8) parameters
    l8_start: str = "2014-05-01"
    l8_end: str = "2014-10-31"
    max_cloud_cover: float = 10.0

    # SAR (Sentinel-1) parameters
    s1_start: str = "2014-06-01"
    s1_end: str = "2018-01-01"
    speckle_filter_radius_px: float = 1.5

    # Random Forest hyperparameters
    rf_num_trees: int = 200
    rf_min_leaf_population: int = 1
    rf_bag_fraction: float = 0.7

    # Sampling
    sample_scale_m: int = 30
    n_background_points: int = 200
    train_fraction: float = 0.7
    random_seed: int = 42

    # Known mineral occurrences (class = 1). REPLACE with verified data.
    positive_sites: Sequence[tuple[float, float]] = field(
        default_factory=lambda: [
            (27.48, -11.66),
            (27.50, -11.70),
            (25.85, -10.72),
            (26.75, -10.95),
        ]
    )

In [6]:
# Define your area of interest here (example: Katanga Copperbelt, DRC)
example_aoi = ee.Geometry.Rectangle([25.0, -12.5, 28.0, -10.0])
cfg = ProspectivityConfig(aoi=example_aoi)

## 4. Optical Data Acquisition & Cloud Masking (Landsat 8 Collection 2 SR)

In [7]:
def build_landsat_mosaic(cfg: ProspectivityConfig) -> ee.Image:
    """Return a cloud-masked, scaled, median Landsat 8 SR composite."""

    def mask_l8_clouds(image: ee.Image) -> ee.Image:
        qa = image.select("QA_PIXEL")
        cloud_shadow_bit = 1 << 4
        cloud_bit = 1 << 3
        mask = (
            qa.bitwiseAnd(cloud_shadow_bit)
            .eq(0)
            .And(qa.bitwiseAnd(cloud_bit).eq(0))
        )
        return image.updateMask(mask)

    collection = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(cfg.aoi)
        .filterDate(cfg.l8_start, cfg.l8_end)
        .filter(ee.Filter.lt("CLOUD_COVER", cfg.max_cloud_cover))
    )

    n_images = collection.size().getInfo()
    if n_images == 0:
        raise ValueError(
            f"No Landsat 8 scenes found for the given AOI/date range "
            f"({cfg.l8_start} to {cfg.l8_end}, cloud cover < {cfg.max_cloud_cover}%). "
            "Widen the date range or relax the cloud filter."
        )
    logger.info("Landsat 8: %d scenes matched filter criteria.", n_images)

    mosaic = (
        collection.map(mask_l8_clouds)
        .median()
        .multiply(2.75e-5)
        .add(-0.2)  # Collection 2 Level-2 SR scale/offset (USGS spec)
        .clip(cfg.aoi)
    )
    return mosaic

## 5. Topographic Derivatives (SRTM DEM)

In [8]:
def build_topographic_layers(cfg: ProspectivityConfig) -> dict[str, ee.Image]:
    """Return DEM, slope, and hillshade clipped to the AOI."""
    dem = ee.Image("USGS/SRTMGL1_003").clip(cfg.aoi)
    slope = ee.Terrain.slope(dem).rename("Slope")
    hillshade = ee.Terrain.hillshade(dem).rename("Hillshade")
    return {"dem": dem.rename("Elevation"), "slope": slope, "hillshade": hillshade}

## 6. SAR Structural Lineament Extraction (Sentinel-1)

In [9]:
def build_lineament_density(cfg: ProspectivityConfig) -> tuple[ee.Image, ee.Image]:
    """Return a directional-gradient lineament density layer from Sentinel-1 VV."""
    s1_collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(cfg.aoi)
        .filterDate(cfg.s1_start, cfg.s1_end)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .select(["VV", "VH"])
    )

    n_images = s1_collection.size().getInfo()
    if n_images == 0:
        raise ValueError(
            f"No Sentinel-1 IW/VV scenes found for {cfg.s1_start} to {cfg.s1_end}."
        )
    logger.info("Sentinel-1: %d scenes matched filter criteria.", n_images)

    s1_mosaic = s1_collection.median().clip(cfg.aoi)
    vv = s1_mosaic.select("VV")

    speckle_filtered = vv.focal_median(
        radius=cfg.speckle_filter_radius_px, kernelType="circle", units="pixels"
    )

    kernels = {
        "N": ee.Kernel.fixed(3, 3, [[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
        "E": ee.Kernel.fixed(3, 3, [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
        "NE": ee.Kernel.fixed(3, 3, [[0, 1, 2], [-1, 0, 1], [-2, -1, 0]]),
        "NW": ee.Kernel.fixed(3, 3, [[2, 1, 0], [1, 0, -1], [0, -1, -2]]),
    }
    gradients = [speckle_filtered.convolve(k).abs() for k in kernels.values()]
    edge_strength = gradients[0].max(gradients[1]).max(gradients[2]).max(gradients[3])
    return edge_strength.rename("Lineament_Density"), s1_mosaic

## 7. Exploration-Specific Spectral Proxies

In [10]:
def build_mineral_indices(mosaic: ee.Image) -> ee.Image:
    """Return band-ratio proxies for hydrothermal alteration and vegetation."""
    b1, b2, b3, b4, b5, b6, b7 = (
        mosaic.select(f"SR_B{i}") for i in (1, 2, 3, 4, 5, 6, 7)
    )

    clay = b5.divide(b7).rename("Clay_Index")          # Phyllosilicates
    ferrous = b5.divide(b4).rename("Ferrous_Iron")      # Iron oxides / gossans
    iron_oxide = b3.divide(b1).rename("Iron_Oxide")     # Ferric iron
    silica = b6.divide(b7).rename("Silica_Index")       # Silicification
    ndvi = b5.subtract(b4).divide(b5.add(b4)).rename("NDVI")

    return ee.Image.cat([clay, ferrous, iron_oxide, silica, ndvi])

## 8. Feature Stack Assembly

In [11]:
def build_feature_stack(cfg: ProspectivityConfig) -> tuple[ee.Image, ee.Image]:
    """Assemble all predictive layers into one multi-band feature image.

    Returns
    -------
    (feature_stack, landsat_mosaic) : tuple of ee.Image
        The stacked predictors, and the source optical mosaic (kept
        separately for true-color visualization).
    """
    mosaic = build_landsat_mosaic(cfg)
    topo = build_topographic_layers(cfg)
    lineament_density, s1_mosaic = build_lineament_density(cfg)
    mineral_indices = build_mineral_indices(mosaic)

    feature_stack = ee.Image.cat(
        [
            lineament_density,
            mineral_indices,
            topo["slope"],
            topo["dem"],
            s1_mosaic.select("VV"),
            s1_mosaic.select("VH"),
        ]
    ).clip(cfg.aoi)

    logger.info(
        "Feature stack assembled with bands: %s", feature_stack.bandNames().getInfo()
    )
    return feature_stack, mosaic

## 9. Training Data, Model Training & Accuracy Assessment

In [12]:
def build_training_data(cfg: ProspectivityConfig) -> ee.FeatureCollection:
    """Merge known-occurrence (class 1) and pseudo-absence (class 0) points.

    Warning
    -------
    Pseudo-absence points are randomly sampled background locations,
    not verified barren ground. Treat model outputs as a relative
    prospectivity ranking, not a validated presence/absence prediction.
    """
    deposits = ee.FeatureCollection(
        [ee.Feature(ee.Geometry.Point([lon, lat]), {"class": 1})
         for lon, lat in cfg.positive_sites]
    )

    background = ee.FeatureCollection.randomPoints(
        region=cfg.aoi, points=cfg.n_background_points, seed=cfg.random_seed
    ).map(lambda f: f.set("class", 0))

    return deposits.merge(background)

In [13]:
def train_and_evaluate(
    feature_stack: ee.Image, cfg: ProspectivityConfig
) -> tuple[ee.Classifier, dict]:
    """Sample the stack, split train/test, train an RF classifier, and
    report held-out accuracy metrics.

    Returns
    -------
    (classifier, metrics) : the trained probability-mode classifier and
        a dict with accuracy, kappa, and a confusion-matrix array.
    """
    training_points = build_training_data(cfg)

    samples = feature_stack.sampleRegions(
        collection=training_points,
        properties=["class"],
        scale=cfg.sample_scale_m,
        geometries=True,
    )

    samples = samples.randomColumn(seed=cfg.random_seed)
    train_set = samples.filter(ee.Filter.lt("random", cfg.train_fraction))
    test_set = samples.filter(ee.Filter.gte("random", cfg.train_fraction))

    n_train = train_set.size().getInfo()
    n_test = test_set.size().getInfo()
    logger.info("Training samples: %d | Held-out test samples: %d", n_train, n_test)
    if n_train == 0 or n_test == 0:
        raise ValueError(
            "Train/test split produced an empty set. Increase "
            "n_background_points or adjust train_fraction."
        )

    # Deterministic classifier for confusion-matrix accuracy assessment
    eval_classifier = ee.Classifier.smileRandomForest(
        numberOfTrees=cfg.rf_num_trees,
        minLeafPopulation=cfg.rf_min_leaf_population,
        bagFraction=cfg.rf_bag_fraction,
        seed=cfg.random_seed,
    ).train(
        features=train_set,
        classProperty="class",
        inputProperties=feature_stack.bandNames(),
    )

    test_classified = test_set.classify(eval_classifier)
    error_matrix = test_classified.errorMatrix("class", "classification")

    metrics = {
        "overall_accuracy": error_matrix.accuracy().getInfo(),
        "kappa": error_matrix.kappa().getInfo(),
        "confusion_matrix": error_matrix.getInfo(),
    }
    logger.info(
        "Held-out accuracy: %.3f | Kappa: %.3f",
        metrics["overall_accuracy"],
        metrics["kappa"],
    )

    # Final probability-mode classifier trained on ALL labeled samples,
    # used for the deployed prospectivity map.
    prob_classifier = ee.Classifier.smileRandomForest(
        numberOfTrees=cfg.rf_num_trees,
        minLeafPopulation=cfg.rf_min_leaf_population,
        bagFraction=cfg.rf_bag_fraction,
        seed=cfg.random_seed,
    ).setOutputMode("probability").train(
        features=samples,
        classProperty="class",
        inputProperties=feature_stack.bandNames(),
    )

    return prob_classifier, metrics

## 10. Run the Pipeline

This executes stages 4–9 end-to-end and prints held-out accuracy metrics.

In [14]:
feature_stack, mosaic = build_feature_stack(cfg)
topo = build_topographic_layers(cfg)
lineament_density, _ = build_lineament_density(cfg)
mineral_indices = build_mineral_indices(mosaic)

classifier, metrics = train_and_evaluate(feature_stack, cfg)
prospectivity_map = feature_stack.classify(classifier).rename("Prospectivity_Probability")
training_points = build_training_data(cfg)

print("Overall accuracy:", metrics["overall_accuracy"])
print("Kappa:", metrics["kappa"])
print("Confusion matrix:", metrics["confusion_matrix"])

Overall accuracy: 0.9714285714285714
Kappa: 0
Confusion matrix: [[68, 0], [2, 0]]


## 11. Visualization

In [15]:
m = geemap.Map()

m.addLayer(
    mosaic, {"bands": ["SR_B4", "SR_B3", "SR_B2"], "min": 0.0, "max": 0.3},
    "Landsat 8 True Color", False,
)
m.addLayer(
    topo["dem"], {"min": 900, "max": 2000,
                   "palette": ["0000ff", "00ff00", "ffff00", "ff0000"]},
    "DEM (SRTM)", False,
)
m.addLayer(
    lineament_density, {"min": 0, "max": 2, "palette": ["black", "magenta"]},
    "Structural Lineaments (SAR)", False,
)
m.addLayer(
    mineral_indices.select("Clay_Index"),
    {"min": 1, "max": 2, "palette": ["blue", "yellow", "red"]},
    "Clay Alteration Index", False,
)

prospectivity_viz = {
    "min": 0.0,
    "max": 1.0,
    "palette": ["0000FF", "00FFFF", "FFFF00", "FF8000", "FF0000"],
}
m.addLayer(prospectivity_map, prospectivity_viz, "ML Prospectivity Map (Random Forest)", True)
m.addLayer(training_points, {"color": "white"}, "Training Points")
m.addLayerControl()
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

## 12. Optional: Export to Google Drive

Uncomment and run to kick off an asynchronous export. Monitor progress with `task.status()` or the [Earth Engine Tasks tab](https://code.earthengine.google.com/tasks).

In [ ]:
# task = ee.batch.Export.image.toDrive(
#     image=prospectivity_map,
#     description="prospectivity_map_export",
#     folder="GEE_exports",
#     region=cfg.aoi,
#     scale=cfg.sample_scale_m,
#     maxPixels=1e13,
# )
# task.start()
# task.status()